# Actividad Autónoma 5 — Unidad 3 Tema 1 — Air Quality (Preprocesamiento + Modelos)

**Nombre:** Pablo Andrés Terán Corrales
**Carrera:** Ciencia de Datos (UNACH)  
**Actividad:** Preprocesamiento y Transformación de Datos + Modelos de Clasificación y Regresión  
**Dataset adjunto:** `Air_Quality.csv` (archivo entregado junto a la actividad)


**Tamaño del dataset:** 52,560 filas × 9 columnas.

**Objetivo del análisis**  
1) Preprocesar datos (nulos, outliers, codificación y escalado manual).  
2) Construir **dos modelos** con scikit-learn:  
   - Clasificación: predecir si el índice de calidad de aire (**AQI**) está “alto” (percentil 75 o superior).  
   - Regresión: predecir el valor numérico de **AQI**.  
3) Comparar resultados y discutir limitaciones y mejoras.  
4) (Reto adicional) Aplicar **feature selection** y observar el cambio en desempeño.

**Criterios del enunciado**: limpieza, codificación, estandarización/normalización manual, visualización exploratoria, modelos de clasificación y regresión, análisis crítico y (opcional) selección de características.

## 0) Librerías y configuración

- Se usa **Pandas** y **NumPy** para el preprocesamiento.
- **No** se usa `sklearn` para escalar (solo para modelos y partición train/test), tal como exige el enunciado.
- Para visualizaciones se usa **Matplotlib**.

In [3]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.feature_selection import SelectKBest, mutual_info_classif, f_regression

import matplotlib.pyplot as plt

# Reproducibilidad
RNG = 42
np.random.seed(RNG)

pd.set_option("display.max_columns", 50)

## 1) Selección y carga del dataset

Se trabaja con el archivo adjunto `Air_Quality.csv`, que contiene mediciones (por hora) de contaminantes y un índice **AQI**.

In [3]:
import pandas as pd

df = pd.read_csv("Air_Quality.csv")
df.head()



,Date,City,CO,NO2,SO2,O3,PM2.5,PM10,AQI
0,2025-01-01 00:00:00+00:00,Brasilia,325.0,21.1,2.5,35.0,15.4,15.6,20.483337
1,2025-01-01 01:00:00+00:00,Brasilia,369.0,20.8,2.7,35.0,15.1,15.3,20.425000
2,2025-01-01 02:00:00+00:00,Brasilia,419.0,20.4,3.0,34.0,15.6,15.8,20.333332
3,2025-01-01 03:00:00+00:00,Brasilia,451.0,20.5,3.1,33.0,16.4,16.6,20.258335
4,2025-01-01 04:00:00+00:00,Brasilia,458.0,22.1,3.0,29.0,17.7,17.8,20.316668


### 1.1) Inspección rápida de tipos y nulos

Antes de limpiar, se revisa:
- Tipos de datos
- Cantidad de valores faltantes (nulos)
- Estadísticos básicos para detectar rangos anómalos

In [ ]:
df.info()

In [ ]:
df.isna().sum().sort_values(ascending=False)

In [ ]:
df.describe(include="all").T

## 2) Preprocesamiento

En esta sección se realiza:
1) Conversión de tipos (fecha a datetime).  
2) Transformación de variables (atributos temporales).  
3) Tratamiento de nulos (imputación).  
4) Tratamiento de outliers (recorte por IQR).  
5) Codificación de categóricas (City → dummies).  
6) Escalado **manual** (estandarización y normalización).

### 2.1) Conversión de `Date` y creación de variables temporales

La columna `Date` viene como texto. Se convierte a datetime y se generan variables con información del tiempo:
- `hour`: hora del día
- `dayofweek`: día de la semana (0=Lunes, 6=Domingo)
- `month`: mes (1-12)

Estas variables suelen ser útiles porque la contaminación puede variar por horarios (tráfico, actividad industrial) y por estacionalidad.

In [ ]:
# Parseo seguro de fecha
df["Date"] = pd.to_datetime(df["Date"], errors="coerce", utc=True)

df["hour"] = df["Date"].dt.hour
df["dayofweek"] = df["Date"].dt.dayofweek
df["month"] = df["Date"].dt.month

# Revisión de conversiones fallidas
df["Date"].isna().sum(), df[["Date","hour","dayofweek","month"]].head()

### 2.2) Visualización exploratoria inicial

Se observan:
- Distribuciones (histogramas) de variables numéricas
- Boxplots para una lectura rápida de dispersión y posibles outliers

Esto ayuda a justificar las decisiones de limpieza y escalado.

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
num_cols

In [ ]:
# Histogramas
df[num_cols].hist(bins=30, figsize=(12, 8))
plt.suptitle("Distribuciones de variables numéricas", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots
plt.figure(figsize=(12, 6))
plt.boxplot([df[c].dropna().values for c in num_cols], labels=num_cols, showfliers=True)
plt.xticks(rotation=45, ha="right")
plt.title("Boxplots de variables numéricas (outliers visibles como puntos)")
plt.tight_layout()
plt.show()

### 2.3) Limpieza de valores nulos (imputación)

Estrategia:
- Para variables numéricas: imputación con **mediana por ciudad** cuando sea posible (captura diferencias de escala entre ciudades).
- Si una ciudad no tiene suficientes datos para una mediana (casos raros), se usa la mediana global.

Se evita eliminar filas masivamente porque el dataset es de series temporales y una eliminación agresiva puede romper continuidad.

In [ ]:
# Nulos antes
nulos_antes = df.isna().sum()

# Columnas numéricas originales de medición (sin incluir las derivadas del tiempo por ahora)
base_numeric = ["CO", "NO2", "SO2", "O3", "PM2.5", "PM10", "AQI", "hour", "dayofweek", "month"]

# Medianas por ciudad (solo para numéricas)
city_medians = df.groupby("City")[base_numeric].median(numeric_only=True)

# Imputación
for col in base_numeric:
    # 1) imputar por mediana de la ciudad
    df[col] = df.apply(
        lambda r: city_medians.loc[r["City"], col] if pd.isna(r[col]) and r["City"] in city_medians.index else r[col],
        axis=1
    )
    # 2) fallback con mediana global si aún queda NaN
    df[col] = df[col].fillna(df[col].median())

# Date: si quedaran NaT (no debería), se eliminan porque no se puede reconstruir el tiempo
df = df.dropna(subset=["Date"])

nulos_despues = df.isna().sum()
pd.DataFrame({"nulos_antes": nulos_antes, "nulos_despues": nulos_despues}).sort_values("nulos_antes", ascending=False)

### 2.4) Tratamiento de outliers con IQR (recorte / clipping)

Para no “borrar” datos, se aplica recorte suave por variable:
- Se calcula IQR = Q3 - Q1
- Límite inferior = Q1 - 1.5·IQR
- Límite superior = Q3 + 1.5·IQR
- Valores fuera del rango se **recortan** al límite más cercano.

Esto conserva el tamaño del dataset y reduce el impacto de valores extremos sobre los modelos lineales.

In [ ]:
def iqr_clip(series: pd.Series, k: float = 1.5):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    low = q1 - k * iqr
    high = q3 + k * iqr
    return series.clip(lower=low, upper=high), (q1, q3, low, high)

clip_info = {}
for col in ["CO", "NO2", "SO2", "O3", "PM2.5", "PM10", "AQI"]:
    df[col], info = iqr_clip(df[col], k=1.5)
    clip_info[col] = info

clip_info_df = pd.DataFrame(
    clip_info, index=["Q1","Q3","Low(Q1-1.5IQR)","High(Q3+1.5IQR)"]
).T
clip_info_df

### 2.5) Codificación de variables categóricas

La columna `City` es categórica. Se convierte a variables dummy (one-hot encoding) con `pd.get_dummies`.

Esto permite que los modelos trabajen con variables numéricas sin imponer un “orden” artificial entre ciudades.

In [ ]:
df_encoded = pd.get_dummies(df, columns=["City"], drop_first=True)  # drop_first para evitar colinealidad exacta
df_encoded.shape, df_encoded.columns[:15]

### 2.6) Estandarización y normalización **manual** (solo Pandas/NumPy)

Se construyen dos escalados, de forma explícita:

**(A) Estandarización (z-score)**  


	 z = (x - media) / desviación_estándar

**(B) Normalización Min-Max**  


	 x_norm = (x - min) / (max - min)

En este notebook se entrenan modelos con **z-score** porque suele funcionar bien en modelos lineales; de todas formas se deja implementado Min-Max para comparación y evidencia.

In [ ]:
# Separar features numéricas originales (ya sin City, porque ya está codificada)
# Excluir Date porque no es numérica modelable directamente; se usan hour/dayofweek/month como alternativa.
feature_cols = [c for c in df_encoded.columns if c not in ["Date"]]

# --- Preparación de variables objetivo (targets) ---
# Clasificación: "High_AQI" = 1 si AQI >= percentil 75 (umbral calculado del propio dataset)
q75 = df_encoded["AQI"].quantile(0.75)
df_encoded["High_AQI"] = (df_encoded["AQI"] >= q75).astype(int)

# Para la regresión: el target será "AQI" (continuo)
# Para evitar fuga de información en modelos, AQI NO debe estar como predictor cuando AQI sea target.
# Entonces: 
# - Clasificación: features sin "AQI" y sin "High_AQI"
# - Regresión: features sin "AQI" y sin "High_AQI"

print("Umbral High_AQI (p75):", float(q75))
df_encoded[["AQI","High_AQI"]].head()

In [ ]:
# Definir matrices X e y para cada tarea
drop_for_models = ["AQI", "High_AQI", "Date"]

X = df_encoded.drop(columns=drop_for_models)
y_clf = df_encoded["High_AQI"].copy()
y_reg = df_encoded["AQI"].copy()

# Identificar columnas numéricas en X (dummies ya son numéricas, pero aquí distinguimos para escalar solo ciertas)
X_num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
X_num_cols[:10], len(X_num_cols)

In [ ]:
# Funciones de escalado manual
def standardize_manual(df_in: pd.DataFrame, cols: list[str]):
    df_out = df_in.copy()
    means = df_out[cols].mean()
    stds = df_out[cols].std(ddof=0).replace(0, 1.0)  # evitar división por cero
    df_out[cols] = (df_out[cols] - means) / stds
    return df_out, means, stds

def minmax_manual(df_in: pd.DataFrame, cols: list[str]):
    df_out = df_in.copy()
    mins = df_out[cols].min()
    maxs = df_out[cols].max()
    denom = (maxs - mins).replace(0, 1.0)
    df_out[cols] = (df_out[cols] - mins) / denom
    return df_out, mins, maxs

X_z, z_means, z_stds = standardize_manual(X, X_num_cols)
X_mm, mm_mins, mm_maxs = minmax_manual(X, X_num_cols)

# Verificación rápida del escalado (z-score debería quedar con media ~0 y std ~1)
pd.DataFrame({
    "mean_after_z": X_z[X_num_cols].mean().round(3),
    "std_after_z": X_z[X_num_cols].std(ddof=0).round(3),
    "min_after_mm": X_mm[X_num_cols].min().round(3),
    "max_after_mm": X_mm[X_num_cols].max().round(3),
}).head()

## 3) Modelo de Clasificación

### 3.1) Elección de la variable objetivo

Se define una clasificación binaria:

- **High_AQI = 1** si `AQI` es mayor o igual al percentil 75 del dataset.
- **High_AQI = 0** en caso contrario.

Ventajas de esta elección:
- Es objetiva y basada en la distribución del propio dataset (no requiere “tablas externas”).
- Genera un problema de clasificación con clases razonablemente balanceadas (aprox. 25% positivos).

> Umbral calculado en el dataset: percentil 75 de AQI.

### 3.2) Train/Test split

Se separa el dataset en entrenamiento y prueba para evaluar generalización.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_z, y_clf, test_size=0.2, random_state=RNG, stratify=y_clf
)

X_train.shape, X_test.shape, y_train.mean().round(3), y_test.mean().round(3)

### 3.3) Entrenamiento del modelo

Se usa **LogisticRegression** como modelo base de clasificación:
- Es un modelo lineal interpretable.
- Es un buen punto de partida para medir la utilidad del preprocesamiento.

Se evalúa con:
- Accuracy
- Matriz de confusión
- Classification report (precision/recall/F1)
- ROC-AUC y curva ROC

In [ ]:
clf = LogisticRegression(max_iter=2000, random_state=RNG)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

print("Accuracy:", acc)
print("ROC-AUC:", auc)
print("\nClassification report:\n", classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
cm

In [ ]:
# Matriz de confusión (visual)
plt.figure(figsize=(5, 4))
plt.imshow(cm, interpolation="nearest")
plt.title("Matriz de confusión — Clasificación High_AQI")
plt.xticks([0,1], ["Pred 0","Pred 1"])
plt.yticks([0,1], ["Real 0","Real 1"])

for (i, j), v in np.ndenumerate(cm):
    plt.text(j, i, str(v), ha="center", va="center")

plt.xlabel("Predicción")
plt.ylabel("Real")
plt.tight_layout()
plt.show()

In [ ]:
# Curva ROC
fpr, tpr, thr = roc_curve(y_test, y_proba)

plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, label=f"ROC (AUC={auc:.3f})")
plt.plot([0,1], [0,1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Curva ROC — Logistic Regression")
plt.legend()
plt.tight_layout()
plt.show()

## 4) Modelo de Regresión

### 4.1) Target de regresión

Se elige **AQI** como variable objetivo continua, porque resume la condición de calidad del aire.
Se eliminó `AQI` de los predictores para evitar fuga de información.

### 4.2) Train/Test split + entrenamiento

Se entrena un modelo de **LinearRegression** y se evalúa con:
- MAE (error absoluto medio)
- MSE (error cuadrático medio)
- R² (proporción de varianza explicada)

Además se grafican:
- Predicción vs valor real
- Distribución de residuos

In [ ]:
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_z, y_reg, test_size=0.2, random_state=RNG
)

reg = LinearRegression()
reg.fit(X_train_r, y_train_r)

y_pred_r = reg.predict(X_test_r)

mae = mean_absolute_error(y_test_r, y_pred_r)
mse = mean_squared_error(y_test_r, y_pred_r)
r2 = r2_score(y_test_r, y_pred_r)

print("MAE:", mae)
print("MSE:", mse)
print("R2 :", r2)

In [ ]:
# Predicción vs Real
plt.figure(figsize=(6, 5))
plt.scatter(y_test_r, y_pred_r, s=10, alpha=0.5)
plt.xlabel("AQI real")
plt.ylabel("AQI predicho")
plt.title("Regresión — AQI real vs AQI predicho")
plt.tight_layout()
plt.show()

In [ ]:
# Residuos
residuals = y_test_r - y_pred_r

plt.figure(figsize=(6, 4))
plt.hist(residuals, bins=40)
plt.title("Distribución de residuos (real - predicho)")
plt.xlabel("Residuo")
plt.ylabel("Frecuencia")
plt.tight_layout()
plt.show()

print("Media del residuo:", float(np.mean(residuals)))

## 5) Reto adicional (+1): Feature Selection y comparación

Se aplica selección de características para observar el cambio de desempeño.

- **Clasificación:** `SelectKBest` con `mutual_info_classif` (información mutua).
- **Regresión:** `SelectKBest` con `f_regression` (relación lineal F-test).

Se compara el desempeño con el modelo “base” (sin selección) usando las mismas particiones.

In [ ]:
# --- Feature Selection: Clasificación ---
k = min(10, X_train.shape[1])  # por seguridad
selector_clf = SelectKBest(score_func=mutual_info_classif, k=k)
selector_clf.fit(X_train, y_train)

X_train_sel = selector_clf.transform(X_train)
X_test_sel  = selector_clf.transform(X_test)

clf_fs = LogisticRegression(max_iter=2000, random_state=RNG)
clf_fs.fit(X_train_sel, y_train)

y_pred_fs = clf_fs.predict(X_test_sel)
y_proba_fs = clf_fs.predict_proba(X_test_sel)[:, 1]

acc_fs = accuracy_score(y_test, y_pred_fs)
auc_fs = roc_auc_score(y_test, y_proba_fs)

# ¿qué features quedaron?
selected_mask = selector_clf.get_support()
selected_features = X_train.columns[selected_mask].tolist()

print("k =", k)
print("Selected features (clasificación):")
for f in selected_features:
    print(" -", f)

print("\nBase Accuracy:", acc, " | FS Accuracy:", acc_fs)
print("Base ROC-AUC :", auc, " | FS ROC-AUC :", auc_fs)

In [ ]:
# --- Feature Selection: Regresión ---
k_r = min(10, X_train_r.shape[1])
selector_reg = SelectKBest(score_func=f_regression, k=k_r)
selector_reg.fit(X_train_r, y_train_r)

X_train_r_sel = selector_reg.transform(X_train_r)
X_test_r_sel  = selector_reg.transform(X_test_r)

reg_fs = LinearRegression()
reg_fs.fit(X_train_r_sel, y_train_r)

y_pred_r_fs = reg_fs.predict(X_test_r_sel)

mae_fs = mean_absolute_error(y_test_r, y_pred_r_fs)
mse_fs = mean_squared_error(y_test_r, y_pred_r_fs)
r2_fs  = r2_score(y_test_r, y_pred_r_fs)

selected_features_r = X_train_r.columns[selector_reg.get_support()].tolist()

print("k =", k_r)
print("Selected features (regresión):")
for f in selected_features_r:
    print(" -", f)

print("\nBase MAE:", mae, " | FS MAE:", mae_fs)
print("Base MSE:", mse, " | FS MSE:", mse_fs)
print("Base R2 :", r2,  " | FS R2 :", r2_fs)

## 6) Análisis comparativo y conclusiones

**(1) Preprocesamiento clave**
- Imputación por mediana (por ciudad) ayudó a evitar pérdida de información.
- Recorte por IQR redujo el efecto de valores extremos sobre modelos lineales.
- Codificación one-hot permitió incorporar la ciudad sin forzar un orden numérico.
- Estandarización z-score facilitó que variables con diferentes escalas (CO, PM2.5, etc.) aporten de forma comparable.

**(2) Clasificación**
- La métrica principal aquí es ROC-AUC (probabilidades) y F1 (balance precisión/recall).
- Si el modelo falla en la clase positiva (High_AQI), podría requerirse:
  - un modelo no lineal (árboles, boosting),
  - ingeniería de variables (lags temporales),
  - balanceo o calibración de umbral.

**(3) Regresión**
- MAE indica el error promedio en unidades de AQI.
- MSE penaliza más los errores grandes.
- R² muestra cuánto del comportamiento del AQI se explica con los predictores disponibles.

**(4) Reto adicional: feature selection**
- Si el desempeño mejora o se mantiene, significa que el modelo puede simplificarse y volverse más estable.
- Si cae, puede indicar que el AQI depende de una combinación más amplia de variables o de relaciones no lineales.

**Limitaciones y mejoras**
- Este análisis trata cada fila como independiente; en series temporales es frecuente que existan dependencias (autocorrelación).
  Una mejora concreta sería crear variables rezagadas (lags) y usar validación respetando el tiempo (train antes, test después).
- También se podría probar un modelo de árbol para capturar no linealidades sin requerir muchas transformaciones.

---
### Referencia del enunciado (archivo adjunto)
Este notebook implementa los puntos solicitados (preprocesamiento, modelos de clasificación y regresión, métricas, visualizaciones y reto adicional de feature selection) conforme al documento de la actividad.

Archivo: `Act_Autónoma_Programacion_2_Unidad_3_Tema_1.pdf`